In [1]:
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
notes_val = pd.read_csv("notes_val_predictions_task3.csv")
ecg_val = pd.read_csv("ecg_val_predictions_task3.csv")
structured_val = pd.read_csv("structured_val_predictions_task3.csv")

display(notes_val.head())
display(ecg_val.head())
display(structured_val.head())

print(len(notes_val))
print(len(ecg_val))
print(len(structured_val))

,sample_id,y_true,pred_prob_notes
0,0,0,0.004867
1,1,0,0.004867
2,2,0,0.004915
3,3,0,0.004925
4,4,0,0.004915


,sample_id,y_true,pred_prob_ecg
0,0,0,0.027183
1,1,0,0.027183
2,2,0,0.027183
3,3,0,0.027183
4,4,0,0.027183


,sample_id,y_true,pred_prob_structured
0,0,0,0.038192
1,1,0,0.040124
2,2,0,0.027487
3,3,0,0.077132
4,4,0,0.021838


18986
18986
6134


In [3]:
val_df = notes_val.merge(
    ecg_val[["sample_id", "pred_prob_ecg"]],
    on="sample_id",
    how="inner"
).merge(
    structured_val[["sample_id", "pred_prob_structured"]],
    on="sample_id",
    how="inner"
)

val_df

,sample_id,y_true,pred_prob_notes,pred_prob_ecg,pred_prob_structured
0,0,0,0.004867,0.027183,0.038192
1,1,0,0.004867,0.027183,0.040124
2,2,0,0.004915,0.027183,0.027487
3,3,0,0.004925,0.027183,0.077132
4,4,0,0.004915,0.027183,0.021838
...,...,...,...,...,...
6129,6129,0,0.004877,0.027183,0.025832
6130,6130,0,0.004886,0.027183,0.058229
6131,6131,0,0.004877,0.027183,0.041599
6132,6132,0,0.004886,0.027183,0.035023


In [4]:
notes_test = pd.read_csv("notes_test_predictions_task3.csv")
ecg_test = pd.read_csv("ecg_test_predictions_task3.csv")
structured_test = pd.read_csv("structured_test_predictions_task3.csv")

display(notes_test.head())
display(ecg_test.head())
display(structured_test.head())

print(len(notes_test))
print(len(ecg_test))
print(len(structured_test))

,sample_id,y_true,pred_prob_notes
0,0,0,0.004915
1,1,0,0.008916
2,2,0,0.009250
3,3,0,0.004877
4,4,0,0.004905


,sample_id,y_true,pred_prob_ecg
0,0,0,0.027183
1,1,0,0.027183
2,2,0,0.027183
3,3,0,0.027183
4,4,0,0.027183


,sample_id,y_true,pred_prob_structured
0,0,0,0.038819
1,1,0,0.033439
2,2,0,0.032558
3,3,0,0.041476
4,4,0,0.031857


18987
18987
6202


In [5]:
test_df = notes_test.merge(
    ecg_test[["sample_id", "pred_prob_ecg"]],
    on="sample_id",
    how="inner"
).merge(
    structured_test[["sample_id", "pred_prob_structured"]],
    on="sample_id",
    how="inner"
)
test_df

,sample_id,y_true,pred_prob_notes,pred_prob_ecg,pred_prob_structured
0,0,0,0.004915,0.027183,0.038819
1,1,0,0.008916,0.027183,0.033439
2,2,0,0.009250,0.027183,0.032558
3,3,0,0.004877,0.027183,0.041476
4,4,0,0.004905,0.027183,0.031857
...,...,...,...,...,...
6197,6197,0,0.004896,0.027183,0.026513
6198,6198,0,0.004886,0.027183,0.096896
6199,6199,0,0.004896,0.027183,0.045594
6200,6200,0,0.004886,0.027183,0.068646


In [6]:
# Feature engineering
def add_meta_features(df):
    eps = 1e-6
    
    # base probs
    s = df["pred_prob_structured"].astype(float)
    n = df["pred_prob_notes"].astype(float)
    e = df["pred_prob_ecg"].astype(float)
    
    # interaction terms
    df["notes_x_ecg"] = n * e
    df["notes_x_struct"] = n * s
    df["ecg_x_struct"] = e * s
    df["notes_x_ecg_x_struct"] = n * e * s
    
    # pairwise differences
    df["struct_minus_notes"] = s - n
    df["struct_minus_ecg"] = s - e
    df["notes_minus_ecg"] = n - e
    
    # summary stats across modalities
    df["prob_mean"] = (s + n + e) / 3.0
    df["prob_max"] = np.maximum(np.maximum(s, n), e)
    df["prob_min"] = np.minimum(np.minimum(s, n), e)
    df["prob_range"] = df["prob_max"] - df["prob_min"]
    
    # agreement / dispersion
    df["prob_std"] = np.std(np.vstack([s, n, e]), axis=0)
    
    # logit transform can help logistic stacking
    df["logit_struct"] = np.log((s + eps) / (1 - s + eps))
    df["logit_notes"] = np.log((n + eps) / (1 - n + eps))
    df["logit_ecg"] = np.log((e + eps) / (1 - e + eps))
    
    # rank-like indicators / thresholds
    df["struct_gt_notes"] = (s > n).astype(int)
    df["struct_gt_ecg"] = (s > e).astype(int)
    df["notes_gt_ecg"] = (n > e).astype(int)
    
    # confidence-style features
    df["struct_conf"] = np.abs(s - 0.5)
    df["notes_conf"] = np.abs(n - 0.5)
    df["ecg_conf"] = np.abs(e - 0.5)
    
    return df

val_df = add_meta_features(val_df.copy())
test_df = add_meta_features(test_df.copy())

# Features
stack_features = [
    # raw probs
    "pred_prob_structured",
    "pred_prob_notes",
    "pred_prob_ecg",
    
    # interactions
    "notes_x_ecg",
    "notes_x_struct",
    "ecg_x_struct",
    "notes_x_ecg_x_struct",
    
    # differences
    "struct_minus_notes",
    "struct_minus_ecg",
    "notes_minus_ecg",
    
    # summaries
    "prob_mean",
    "prob_max",
    "prob_min",
    "prob_range",
    "prob_std",
    
    # logits
    "logit_struct",
    "logit_notes",
    "logit_ecg",
    
    # comparisons
    "struct_gt_notes",
    "struct_gt_ecg",
    "notes_gt_ecg",
    
    # confidence
    "struct_conf",
    "notes_conf",
    "ecg_conf",
]

X_val = val_df[stack_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_val = val_df["y_true"].astype(int)

X_test = test_df[stack_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_test = test_df["y_true"].astype(int)

# Logistic stacking model
meta_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.2,
        C=0.5,
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    ))
])

meta_model.fit(X_val, y_val)

# Predict + evaluate
test_df["pred_logistic_ensemble"] = meta_model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, test_df["pred_logistic_ensemble"])
auprc = average_precision_score(y_test, test_df["pred_logistic_ensemble"])

print("Logistic multimodal test AUROC:", round(auc, 6))
print("Logistic multimodal test AUPRC:", round(auprc, 6))

# Inspect learned weights
lr = meta_model.named_steps["lr"]
coefs = pd.DataFrame({
    "feature": stack_features,
    "coef": lr.coef_[0]
}).sort_values("coef", ascending=False)

print("\nTop positive coefficients:")
print(coefs.head(10).to_string(index=False))

print("\nTop negative coefficients:")
print(coefs.tail(10).to_string(index=False))

Logistic multimodal test AUROC: 0.564609
Logistic multimodal test AUPRC: 0.032371

Top positive coefficients:
             feature     coef
         logit_notes 1.099585
         struct_conf 0.228062
          notes_conf 0.180764
  struct_minus_notes 0.160882
        ecg_x_struct 0.149075
    struct_minus_ecg 0.149075
pred_prob_structured 0.149075
           prob_mean 0.136898
        logit_struct 0.016717
        notes_gt_ecg 0.000000

Top negative coefficients:
             feature      coef
          prob_range -0.044050
            prob_max -0.054238
      notes_x_struct -0.070344
notes_x_ecg_x_struct -0.070344
       struct_gt_ecg -0.161121
     pred_prob_notes -0.180764
            prob_min -0.180764
         notes_x_ecg -0.180764
     notes_minus_ecg -0.180764
            prob_std -0.226728
